Code for Stocks chosen by screener and the MSRP and MVP portfolio on it

In [ ]:
# ======================================
# ✅ 1) Libraries & Install
# ======================================
!pip install PyPortfolioOpt yfinance seaborn matplotlib

import yfinance as yf
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import seaborn as sns
import matplotlib.pyplot as plt

from pypfopt.expected_returns import mean_historical_return
from pypfopt.risk_models import sample_cov
from pypfopt.efficient_frontier import EfficientFrontier

# ======================================
# ✅ 2) Download Data
# ======================================
stocks = ['AIIL.NS', 'BALUFORGE.NS', 'RMDRIP.NS', 'GOKULAGRO.NS', 'CONSOFINVT.NS']

data = yf.download(stocks, start="2025-05-01", end="2025-05-30", auto_adjust=True)
close = data["Close"][stocks]
returns = close.pct_change(fill_method=None).dropna(how="all")

# ======================================
# ✅ 3) Manual Mean & Cov
# ======================================
mu_annual = returns.mean() * 252
cov_annual = returns.cov() * 252

print("\n✅ Manual Annualized Expected Returns:")
print(mu_annual)

print("\n✅ Manual Annualized Covariance Matrix:")
print(cov_annual)

# ======================================
# ✅ 4) Manual MVP
# ======================================
n = len(stocks)
w0 = np.ones(n) / n

def portfolio_variance(w, cov_matrix):
    return w.T @ cov_matrix @ w

constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
bounds = [(0.0, 1.0) for _ in range(n)]

result_mvp = minimize(
    fun=portfolio_variance,
    x0=w0,
    args=(cov_annual,),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)
w_mvp_manual = pd.Series(result_mvp.x, index=stocks)

print("\n✅ Manual MVP Weights:")
print(w_mvp_manual.round(4))

# ======================================
# ✅ 5) Manual MSRP
# ======================================
risk_free_rate = 0.0636
excess_mu = mu_annual - risk_free_rate

def negative_sharpe(w, excess_mu, cov_matrix):
    port_return = np.dot(w, excess_mu)
    port_std = np.sqrt(w.T @ cov_matrix @ w)
    return -port_return / port_std

result_msrp = minimize(
    fun=negative_sharpe,
    x0=w0,
    args=(excess_mu, cov_annual),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)
w_msrp_manual = pd.Series(result_msrp.x, index=stocks)

print("\n✅ Manual MSRP Weights:")
print(w_msrp_manual.round(4))

# ======================================
# ✅ 6) PyPortfolioOpt version
# ======================================
mu_pypfopt = mean_historical_return(close, compounding=False)
S_pypfopt = sample_cov(close)

ef_mvp = EfficientFrontier(mu_pypfopt, S_pypfopt)
w_mvp_pypfopt = pd.Series(ef_mvp.min_volatility())
w_mvp_pypfopt = pd.Series(ef_mvp.clean_weights())

ef_msrp = EfficientFrontier(mu_pypfopt, S_pypfopt)
w_msrp_pypfopt = pd.Series(ef_msrp.max_sharpe(risk_free_rate=risk_free_rate))
w_msrp_pypfopt = pd.Series(ef_msrp.clean_weights())

print("\n✅ PyPortfolioOpt MVP Weights:")
print(w_mvp_pypfopt.round(4))

print("\n✅ PyPortfolioOpt MSRP Weights:")
print(w_msrp_pypfopt.round(4))

# ======================================
# ✅ 7) Compare Weights Plots
# ======================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
w_mvp_manual.plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('MVP (Manual)')
axes[0].set_ylim(0, 1)

w_mvp_pypfopt.plot(kind='bar', ax=axes[1], color='lightgreen')
axes[1].set_title('MVP (PyPortfolioOpt)')
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
w_msrp_manual.plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('MSRP (Manual)')
axes[0].set_ylim(0, 1)

w_msrp_pypfopt.plot(kind='bar', ax=axes[1], color='lightgreen')
axes[1].set_title('MSRP (PyPortfolioOpt)')
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

# ======================================
# ✅ 8) Heatmaps
# ======================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(cov_annual, annot=True, fmt=".4f", cmap="Reds", ax=axes[0])
axes[0].set_title("Covariance Matrix (Manual)")

sns.heatmap(S_pypfopt, annot=True, fmt=".4f", cmap="Blues", ax=axes[1])
axes[1].set_title("Covariance Matrix (PyPortfolioOpt)")
plt.tight_layout()
plt.show()

# ======================================
# ✅ 9) Expected Return, Risk & Sharpe
# ======================================
def expected_portfolio_return(weights, mu):
    return np.dot(weights, mu)

def portfolio_std(weights, cov_matrix):
    return np.sqrt(weights.T @ cov_matrix @ weights)

# Manual
manual_mvp_return = expected_portfolio_return(w_mvp_manual, mu_annual)
manual_mvp_std = portfolio_std(w_mvp_manual, cov_annual)
manual_mvp_sharpe = (manual_mvp_return - risk_free_rate) / manual_mvp_std

manual_msrp_return = expected_portfolio_return(w_msrp_manual, mu_annual)
manual_msrp_std = portfolio_std(w_msrp_manual, cov_annual)
manual_msrp_sharpe = (manual_msrp_return - risk_free_rate) / manual_msrp_std

# PyPortfolioOpt
pypfopt_mvp_return = expected_portfolio_return(w_mvp_pypfopt.values, mu_pypfopt)
pypfopt_mvp_std = portfolio_std(w_mvp_pypfopt.values, S_pypfopt)
pypfopt_mvp_sharpe = (pypfopt_mvp_return - risk_free_rate) / pypfopt_mvp_std

pypfopt_msrp_return = expected_portfolio_return(w_msrp_pypfopt.values, mu_pypfopt)
pypfopt_msrp_std = portfolio_std(w_msrp_pypfopt.values, S_pypfopt)
pypfopt_msrp_sharpe = (pypfopt_msrp_return - risk_free_rate) / pypfopt_msrp_std

print("\n✅ Expected Annual Returns, Risks & Sharpe Ratios:")
print(f"Manual MVP: Return={manual_mvp_return:.4%} | Risk={manual_mvp_std:.4%} | Sharpe={manual_mvp_sharpe:.4f}")
print(f"Manual MSRP: Return={manual_msrp_return:.4%} | Risk={manual_msrp_std:.4%} | Sharpe={manual_msrp_sharpe:.4f}")
print(f"PyPortfolioOpt MVP: Return={pypfopt_mvp_return:.4%} | Risk={pypfopt_mvp_std:.4%} | Sharpe={pypfopt_mvp_sharpe:.4f}")
print(f"PyPortfolioOpt MSRP: Return={pypfopt_msrp_return:.4%} | Risk={pypfopt_msrp_std:.4%} | Sharpe={pypfopt_msrp_sharpe:.4f}")

# ======================================
# ✅ 10) EF + CAL Plot with MVP & MSRP
# ======================================
def plot_efficient_frontier_and_cal_with_stocks(mu, cov_matrix, ef_msrp, ef_mvp, risk_free_rate, stock_returns, stock_risks, stock_labels):
    ef = EfficientFrontier(mu, cov_matrix)
    max_return = mu.max()
    target_returns = np.linspace(mu.min(), max_return * 0.99, 50)
    risks, rets = [], []

    for r in target_returns:
        ef = EfficientFrontier(mu, cov_matrix)
        ef.efficient_return(target_return=r)
        perf = ef.portfolio_performance()
        risks.append(perf[1])
        rets.append(perf[0])

    # MSRP point
    ret_msrp, risk_msrp, sharpe_msrp = ef_msrp.portfolio_performance(risk_free_rate=risk_free_rate)
    # MVP point
    ret_mvp, risk_mvp, sharpe_mvp = ef_mvp.portfolio_performance(risk_free_rate=risk_free_rate)

    # CAL
    slope = sharpe_msrp
    cal_x = np.linspace(0, max(risks)*1.2, 100)
    cal_y = risk_free_rate + slope * cal_x

    plt.figure(figsize=(10, 7))
    plt.plot(risks, rets, 'b--', label='Efficient Frontier')
    plt.plot(cal_x, cal_y, 'r-', label=f'CAL | MSRP Sharpe: {sharpe_msrp:.4f}')
    plt.scatter(risk_msrp, ret_msrp, marker='*', color='gold', s=200, label='Max Sharpe Portfolio')
    plt.scatter(risk_mvp, ret_mvp, marker='D', color='purple', s=100, label='Min Vol Portfolio')

    for i in range(len(stock_labels)):
        plt.scatter(stock_risks[i], stock_returns[i], marker='o', color='black')
        plt.text(stock_risks[i]+0.001, stock_returns[i]+0.001, stock_labels[i], fontsize=9)

    plt.xlabel('Annualized Volatility (Risk)')
    plt.ylabel('Annualized Expected Return')
    plt.title('Efficient Frontier & CAL with Stocks')
    plt.legend()
    plt.show()

    print(f"\n✅ MSRP Sharpe: {sharpe_msrp:.4f} | Return: {ret_msrp:.4%} | Risk: {risk_msrp:.4%}")
    print(f"✅ MVP Sharpe: {sharpe_mvp:.4f} | Return: {ret_mvp:.4%} | Risk: {risk_mvp:.4%}")

# Individual stocks
stock_returns = mu_annual.values
stock_risks = returns.std().values * np.sqrt(252)
stock_labels = stocks

plot_efficient_frontier_and_cal_with_stocks(
    mu_pypfopt,
    S_pypfopt,
    ef_msrp,
    ef_mvp,
    risk_free_rate,
    stock_returns,
    stock_risks,
    stock_labels
)


Code for Star Investor MSRP and MVP


Investors portfolio with sharpe

In [ ]:
# ======================================
# ✅ 1) Install & Import Libraries
# ======================================
!pip install PyPortfolioOpt yfinance seaborn matplotlib

import yfinance as yf
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import seaborn as sns
import matplotlib.pyplot as plt

from pypfopt.expected_returns import mean_historical_return
from pypfopt.risk_models import sample_cov
from pypfopt.efficient_frontier import EfficientFrontier

# ======================================
# ✅ 2) Download Data
# ======================================
stocks = [
    "ATULAUTO.NS", "TAC.NS", "INNOVATORS.NS", "AFFORDABLE.NS",
    "SUDARSCHEM.NS", "REPRO.NS", "PALREDTEC.NS", "IRIS.NS",
    "INDOSTAR.NS", "GLOBALVECT.NS", "AGI.NS", "FLUOROCHEM.NS",
    "PRATAAP.NS", "NIYOGIN.NS", "RELIGARE.NS", "QUESS.NS"
]

data = yf.download(stocks, start="2025-05-01", end="2025-05-30", auto_adjust=True)
close = data["Close"]

# Calculate daily returns
returns = close.pct_change().dropna(how="all")

# ======================================
# ✅ 3) Drop tickers with invalid data
# ======================================
valid_returns = returns.dropna(axis=1, how="any")
valid_stocks = valid_returns.columns.tolist()
print(f"✅ Valid stocks: {valid_stocks}")

close = close[valid_stocks]
mu_annual = valid_returns.mean() * 252
cov_annual = valid_returns.cov() * 252

print("\nManual Annualized Expected Returns:")
print(mu_annual)

print("\nManual Annualized Covariance Matrix:")
print(cov_annual)

# ======================================
# ✅ 4) Manual MVP
# ======================================
n = len(valid_stocks)
w0 = np.ones(n) / n

def portfolio_variance(w, cov_matrix):
    return w.T @ cov_matrix @ w

constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
bounds = [(0.0, 1.0) for _ in range(n)]

result_mvp = minimize(
    fun = portfolio_variance,
    x0 = w0,
    args = (cov_annual,),
    method = 'SLSQP',
    bounds = bounds,
    constraints = constraints
)
w_mvp_manual = pd.Series(result_mvp.x, index=valid_stocks)
print("\nManual MVP Weights:")
print(w_mvp_manual.round(4))

# ======================================
# ✅ 5) Manual MSRP
# ======================================
risk_free_rate = 0.0636
excess_mu = mu_annual - risk_free_rate

def negative_sharpe(w, excess_mu, cov_matrix):
    port_return = np.dot(w, excess_mu)
    port_std = np.sqrt(w.T @ cov_matrix @ w)
    return -port_return / port_std

result_msrp = minimize(
    fun = negative_sharpe,
    x0 = w0,
    args = (excess_mu, cov_annual),
    method = 'SLSQP',
    bounds = bounds,
    constraints = constraints
)
w_msrp_manual = pd.Series(result_msrp.x, index=valid_stocks)
print("\nManual MSRP Weights:")
print(w_msrp_manual.round(4))

# ======================================
# ✅ 6) PyPortfolioOpt version
# ======================================
mu_pypfopt = mean_historical_return(close, compounding=False)
S_pypfopt = sample_cov(close)

ef_mvp = EfficientFrontier(mu_pypfopt, S_pypfopt)
w_mvp_pypfopt = ef_mvp.min_volatility()
w_mvp_pypfopt = pd.Series(ef_mvp.clean_weights())

ef_msrp = EfficientFrontier(mu_pypfopt, S_pypfopt)
w_msrp_pypfopt = ef_msrp.max_sharpe(risk_free_rate=risk_free_rate)
w_msrp_pypfopt = pd.Series(ef_msrp.clean_weights())

print("\nPyPortfolioOpt MVP Weights:")
print(w_mvp_pypfopt.round(4))

print("\nPyPortfolioOpt MSRP Weights:")
print(w_msrp_pypfopt.round(4))

# ======================================
# ✅ 7) Compute Metrics for ALL
# ======================================
def portfolio_performance(weights, mu, cov, risk_free_rate=0.0):
    exp_return = np.dot(weights, mu)
    volatility = np.sqrt(weights.T @ cov @ weights)
    sharpe = (exp_return - risk_free_rate) / volatility
    return exp_return, volatility, sharpe

# Manual
manual_mvp_perf = portfolio_performance(w_mvp_manual, mu_annual, cov_annual, risk_free_rate)
manual_msrp_perf = portfolio_performance(w_msrp_manual, mu_annual, cov_annual, risk_free_rate)

# PyPortfolioOpt
w_mvp_pypfopt_arr = np.array(list(w_mvp_pypfopt.values))
w_msrp_pypfopt_arr = np.array(list(w_msrp_pypfopt.values))

pypfopt_mvp_perf = portfolio_performance(w_mvp_pypfopt_arr, mu_pypfopt, S_pypfopt, risk_free_rate)
pypfopt_msrp_perf = portfolio_performance(w_msrp_pypfopt_arr, mu_pypfopt, S_pypfopt, risk_free_rate)

# ======================================
# ✅ 8) Print All Shapes & Metrics
# ======================================
print("\n📊 PORTFOLIO PERFORMANCE:")

print(f"\nManual MVP:")
print(f"  Expected Return: {manual_mvp_perf[0]:.4%}")
print(f"  Volatility:      {manual_mvp_perf[1]:.4%}")
print(f"  Sharpe Ratio:    {manual_mvp_perf[2]:.4f}")

print(f"\nManual MSRP:")
print(f"  Expected Return: {manual_msrp_perf[0]:.4%}")
print(f"  Volatility:      {manual_msrp_perf[1]:.4%}")
print(f"  Sharpe Ratio:    {manual_msrp_perf[2]:.4f}")

print(f"\nPyPortfolioOpt MVP:")
print(f"  Expected Return: {pypfopt_mvp_perf[0]:.4%}")
print(f"  Volatility:      {pypfopt_mvp_perf[1]:.4%}")
print(f"  Sharpe Ratio:    {pypfopt_mvp_perf[2]:.4f}")

print(f"\nPyPortfolioOpt MSRP:")
print(f"  Expected Return: {pypfopt_msrp_perf[0]:.4%}")
print(f"  Volatility:      {pypfopt_msrp_perf[1]:.4%}")
print(f"  Sharpe Ratio:    {pypfopt_msrp_perf[2]:.4f}")

# ======================================
# ✅ 9) PLOTS
# ======================================

# ✅ Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(valid_returns.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix Heatmap")
plt.show()

# ✅ Bar plot for PyPortfolioOpt MSRP Weights
w_msrp_pypfopt_nonzero = w_msrp_pypfopt[w_msrp_pypfopt > 0]
w_msrp_pypfopt_nonzero.sort_values().plot(kind="barh", figsize=(8,6))
plt.title("PyPortfolioOpt MSRP Portfolio Weights")
plt.xlabel("Weight")
plt.show()

# ✅ Capital Allocation Line
msrp_return, msrp_volatility, msrp_sharpe = pypfopt_msrp_perf

plt.figure(figsize=(10, 6))
plt.scatter(0, risk_free_rate, color="green", label="Risk-Free Rate")
plt.scatter(msrp_volatility, msrp_return, color="red", label="MSRP Portfolio")

# CAL Line
x = np.linspace(0, msrp_volatility + 0.05, 100)
cal = risk_free_rate + msrp_sharpe * x
plt.plot(x, cal, label="Capital Allocation Line", color="blue")

plt.title("Capital Allocation Line (CAL)")
plt.xlabel("Portfolio Volatility (Risk)")
plt.ylabel("Portfolio Expected Return")
plt.legend()
plt.show()


Code for Screener stocks based on equal , proportional market cap and inverse market cap portfolio

In [ ]:
# ✅ Step 1: Install yfinance if not already installed
# !pip install yfinance

# ✅ Step 2: Imports
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ✅ Step 3: Define NSE/BSE tickers
tickers = ['AIIL.BO', 'BALUFORGE.BO', 'RMDRIP.NS', 'GOKULAGRO.BO', 'CONSOFINVT.NS']

# ✅ Step 4: Fetch shares outstanding (free float) with fallback & logging
float_shares = {}

print("\n📥 Fetching floatShares and sharesOutstanding:")

for ticker in tickers:
    try:
        info = yf.Ticker(ticker).info
        float_share = info.get("floatShares", None)
        shares_out = info.get("sharesOutstanding", None)

        print(f"\n📊 {ticker}:")
        print(f"   🟢 Float Shares: {float_share}")
        print(f"   🔵 Shares Outstanding: {shares_out}")

        if float_share:
            float_shares[ticker] = float_share
        elif shares_out:
            print(f"⚠️ Using sharesOutstanding as fallback for {ticker}")
            float_shares[ticker] = shares_out
        else:
            print(f"❌ No share data found for {ticker}, skipping.")

    except Exception as e:
        print(f"❌ Error fetching data for {ticker}: {e}")

# ✅ Step 5: Download historical price data
data = yf.download(
    tickers=list(float_shares.keys()),
    start='2025-05-01',
    end='2025-05-30',
    interval='1d',
    group_by='ticker',
    auto_adjust=True
)

# ✅ Step 6: Extract adjusted closing prices
adj_close = pd.DataFrame()
latest_prices = {}

for ticker in float_shares:
    if ticker in data.columns.levels[0]:
        adj_close[ticker] = data[ticker]['Close']
        prices = adj_close[ticker].dropna()
        if not prices.empty:
            latest_prices[ticker] = prices.iloc[-1]
    else:
        print(f"⚠️ No price data found for {ticker}")

# ✅ Step 7: Daily returns & total market cap
daily_returns_all = adj_close.pct_change().dropna()
risk_free_rate_daily = 0.0636 / 252

market_caps = {
    ticker: latest_prices[ticker] * float_shares[ticker]
    for ticker in latest_prices
}

total_market_cap = sum(market_caps.values())
print(f"\n💰 Total Free-Float Market Capitalization: ₹{total_market_cap:,.2f}")

# ✅ Step 8: Individual performance analysis
results = []
for ticker in adj_close.columns:
    prices = adj_close[ticker].dropna()
    if prices.empty:
        continue

    start_price = prices.iloc[0]
    end_price = prices.iloc[-1]
    total_return = ((end_price - start_price) / start_price) * 100

    daily_returns = prices.pct_change().dropna()
    daily_std_dev = daily_returns.std()
    annualized_std_dev = daily_std_dev * np.sqrt(252) * 100
    sharpe_ratio = np.sqrt(252) * (daily_returns.mean() - risk_free_rate_daily) / daily_std_dev

    results.append({
        'Ticker': ticker,
        'Total Return (%)': total_return,
        'Daily Std Dev (%)': daily_std_dev * 100,
        'Annualized Std Dev (%)': annualized_std_dev,
        'Sharpe Ratio': sharpe_ratio
    })

# ✅ Step 9: Portfolio performance
n_days = (daily_returns_all.index[-1] - daily_returns_all.index[0]).days

# Equal-weighted
portfolio_equal_daily = daily_returns_all.mean(axis=1)
portfolio_equal_cum = (1 + portfolio_equal_daily).cumprod()

equal_total_return = portfolio_equal_cum.iloc[-1] - 1
equal_annualized_return = ((1 + equal_total_return) ** (365 / n_days) - 1) * 100
sharpe_equal = np.sqrt(252) * (portfolio_equal_daily - risk_free_rate_daily).mean() / portfolio_equal_daily.std()

# Market cap weighted
weights_mcap = {k: v / total_market_cap for k, v in market_caps.items()}
df_weights_mcap = pd.Series(weights_mcap)
portfolio_mcap_daily = (daily_returns_all * df_weights_mcap).sum(axis=1)
portfolio_mcap_cum = (1 + portfolio_mcap_daily).cumprod()

mcap_total_return = portfolio_mcap_cum.iloc[-1] - 1
mcap_annualized_return = ((1 + mcap_total_return) ** (365 / n_days) - 1) * 100
sharpe_mcap = np.sqrt(252) * (portfolio_mcap_daily - risk_free_rate_daily).mean() / portfolio_mcap_daily.std()

# Inverse Market Cap weighted
inv_weights = {k: 1/v for k, v in market_caps.items()}
inv_total = sum(inv_weights.values())
weights_inverse = {k: v / inv_total for k, v in inv_weights.items()}
df_weights_inverse = pd.Series(weights_inverse)
portfolio_inv_daily = (daily_returns_all * df_weights_inverse).sum(axis=1)
portfolio_inv_cum = (1 + portfolio_inv_daily).cumprod()

inv_total_return = portfolio_inv_cum.iloc[-1] - 1
inv_annualized_return = ((1 + inv_total_return) ** (365 / n_days) - 1) * 100
sharpe_inverse = np.sqrt(252) * (portfolio_inv_daily - risk_free_rate_daily).mean() / portfolio_inv_daily.std()

# ✅ Step 10: Display summary
results_df = pd.DataFrame(results)
summary = pd.DataFrame([{
    'Ticker': 'Average',
    'Total Return (%)': results_df['Total Return (%)'].mean(),
    'Daily Std Dev (%)': results_df['Daily Std Dev (%)'].mean(),
    'Annualized Std Dev (%)': results_df['Annualized Std Dev (%)'].mean(),
    'Sharpe Ratio': results_df['Sharpe Ratio'].mean()
}])

final_df = pd.concat([results_df, summary], ignore_index=True)

print("\n📊 Summary of Stock Performance")
print(final_df)

print(f"\n✅ Equal-Weight Return: {equal_annualized_return:.2f}% | Sharpe: {sharpe_equal:.2f}")
print(f"✅ Market Cap Weighted Return: {mcap_annualized_return:.2f}% | Sharpe: {sharpe_mcap:.2f}")
print(f"✅ Inverse Market Cap Return: {inv_annualized_return:.2f}% | Sharpe: {sharpe_inverse:.2f}")

# ✅ Save output
final_df.to_csv("freefloat_performance_summary.csv", index=False)
print("\n✅ Saved as freefloat_performance_summary.csv")

# ✅ Step 11: Add function to plot portfolio performance
def plot_portfolio_performance(portfolios: dict, title: str = "Portfolio Performance Over Time", save_path: str = None):
    """
    Plots cumulative performance of portfolios over time.

    Args:
        portfolios (dict): {portfolio_name: pd.Series of cumulative returns}
        title (str): Title of the plot.
        save_path (str): Path to save the plot image (optional).
    """
    plt.figure(figsize=(12, 7))
    for name, series in portfolios.items():
        plt.plot(series.index, series, label=name)

    plt.title(title, fontsize=16)
    plt.xlabel("Date", fontsize=14)
    plt.ylabel("Cumulative Return (Growth of ₹1)", fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✅ Plot saved to {save_path}")

    plt.show()

# ✅ Step 12: Call the plot function with your portfolios
portfolios = {
    'Equal Weighted': portfolio_equal_cum,
    'Market Cap Weighted': portfolio_mcap_cum,
    'Inverse Market Cap Weighted': portfolio_inv_cum
}

plot_portfolio_performance(
    portfolios,
    title="Free-Float Portfolio Cumulative Performance",
    save_path="freefloat_portfolio_performance.png"
)

# ✅ Optional: Save cumulative returns for further analysis
cum_returns_df = pd.DataFrame({
    'Equal Weighted': portfolio_equal_cum,
    'Market Cap Weighted': portfolio_mcap_cum,
    'Inverse Market Cap Weighted': portfolio_inv_cum
})
cum_returns_df.to_csv("freefloat_portfolio_cumulative_returns.csv")
print("\n✅ Saved cumulative returns as freefloat_portfolio_cumulative_returns.csv")


Backtest for investors portfolio for equal market cap and inverse market cap portfolio


In [ ]:
# ✅ Step 1: Install yfinance if not already installed
# !pip install yfinance

# ✅ Step 2: Imports
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ✅ Step 3: Define base tickers
base_tickers = [
    "ATULAUTO", "TAC", "INNOVATORS", "AFFORDABLE",
    "SUDARSCHEM", "REPRO", "PALREDTEC", "IRIS",
    "INDOSTAR", "GLOBALVECT", "AGI", "FLUOROCHEM",
    "PRATAAP", "NIYOGIN", "RELIGARE", "QUESS"
]

exchanges = [".NS", ".BO"]  # NSE preferred, fallback to BSE
float_shares = {}
final_tickers = {}

# ✅ Step 4: Fetch float shares with fallback
print("\n📥 Fetching floatShares and sharesOutstanding (with BSE fallback):")

for base in base_tickers:
    found = False
    for ext in exchanges:
        ticker = base + ext
        try:
            info = yf.Ticker(ticker).info
            float_share = info.get("floatShares")
            shares_out = info.get("sharesOutstanding")

            if float_share or shares_out:
                final_tickers[base] = ticker
                float_shares[ticker] = float_share if float_share else shares_out
                print(f"\n📊 {ticker}:")
                print(f"   🟢 Float Shares: {float_share}")
                print(f"   🔵 Shares Outstanding: {shares_out}")
                if not float_share:
                    print(f"⚠️ Using sharesOutstanding as fallback for {ticker}")
                found = True
                break
            else:
                print(f"⚠️ No float/share data for {ticker}, trying fallback.")
        except Exception as e:
            print(f"❌ Error fetching {ticker}: {e}")
    if not found:
        print(f"❌ No valid share data found for {base}, skipping.")

# ✅ Step 5: Download price data
valid_tickers = list(float_shares.keys())
if not valid_tickers:
    raise ValueError("❌ No valid tickers found with share data.")

print("\n📥 Downloading historical price data...")
data = yf.download(
    tickers=valid_tickers,
    start='2025-05-01',
    end='2025-05-30',
    interval='1d',
    group_by='ticker',
    auto_adjust=True,
    threads=True
)

# ✅ Step 6: Extract closing prices
adj_close = pd.DataFrame()
latest_prices = {}

for ticker in valid_tickers:
    if ticker in data.columns.levels[0]:
        adj_close[ticker] = data[ticker]['Close']
        prices = adj_close[ticker].dropna()
        if not prices.empty:
            latest_prices[ticker] = prices.iloc[-1]
    else:
        print(f"⚠️ No price data found for {ticker}")

# ✅ Step 7: Daily returns & market cap
adj_close = adj_close.dropna(axis=1, how='all')
daily_returns_all = adj_close.pct_change(fill_method=None).dropna()
risk_free_rate = 0.0636  # annual

market_caps = {
    ticker: latest_prices[ticker] * float_shares[ticker]
    for ticker in adj_close.columns
}
total_market_cap = sum(market_caps.values())

# ✅ Step 8: Stock-wise metrics
results = []
n_days = (daily_returns_all.index[-1] - daily_returns_all.index[0]).days

for ticker in adj_close.columns:
    prices = adj_close[ticker].dropna()
    if prices.empty:
        continue

    start_price = prices.iloc[0]
    end_price = prices.iloc[-1]
    total_return = ((end_price - start_price) / start_price) * 100

    daily_returns = prices.pct_change(fill_method=None).dropna()
    daily_std_dev = daily_returns.std()
    annualized_volatility = daily_std_dev * np.sqrt(252)
    annualized_return = ((1 + total_return / 100) ** (365 / n_days)) - 1

    sharpe_ratio = (annualized_return - risk_free_rate) / annualized_volatility if annualized_volatility != 0 else np.nan

    results.append({
        'Ticker': ticker,
        'Total Return (%)': total_return,
        'Daily Std Dev (%)': daily_std_dev * 100,
        'Annualized Std Dev (%)': annualized_volatility * 100,
        'Sharpe Ratio': sharpe_ratio
    })

results_df = pd.DataFrame(results)

# ✅ Step 9: Portfolio Return — Weighted Returns Method
returns_series = results_df.set_index('Ticker')['Total Return (%)'] / 100
weights_equal = pd.Series(1 / len(returns_series), index=returns_series.index)
weights_mcap = pd.Series({k: v / total_market_cap for k, v in market_caps.items()})
inv_weights_raw = {k: 1 / v for k, v in market_caps.items()}
inv_total = sum(inv_weights_raw.values())
weights_inverse = pd.Series({k: w / inv_total for k, w in inv_weights_raw.items()})

# Calculate total and annualized returns
portfolio_returns = {
    'Equal Weight': weights_equal,
    'Market Cap Weight': weights_mcap,
    'Inverse Market Cap': weights_inverse
}
portfolio_results = {}

for label, weights in portfolio_returns.items():
    total_return = (returns_series * weights).sum()
    annualized_return = ((1 + total_return) ** (365 / n_days) - 1)
    volatility = (daily_returns_all * weights).sum(axis=1).std() * np.sqrt(252)
    sharpe_ratio = (annualized_return - risk_free_rate) / volatility if volatility != 0 else np.nan
    portfolio_results[label] = (annualized_return * 100, sharpe_ratio)

# ✅ Step 10: Display Results
summary = pd.DataFrame([{
    'Ticker': 'Average (All)',
    'Total Return (%)': results_df['Total Return (%)'].mean(),
    'Daily Std Dev (%)': results_df['Daily Std Dev (%)'].mean(),
    'Annualized Std Dev (%)': results_df['Annualized Std Dev (%)'].mean(),
    'Sharpe Ratio': results_df['Sharpe Ratio'].mean()
}])
final_df = pd.concat([results_df, summary], ignore_index=True)

print("\n📊 Summary of Stock Performance")
print(final_df)

for label, (ret, sharpe) in portfolio_results.items():
    print(f"✅ {label} Return: {ret:.2f}% | Sharpe: {sharpe:.2f}")

# ✅ Save output
final_df.to_csv("freefloat_performance_summary.csv", index=False)
print("\n✅ Saved as freefloat_performance_summary.csv")

# ✅ Visualization: Simulated Growth Using Total Returns
dates = pd.date_range(start='2024-04-01', end='2025-03-31')
portfolio_growth = pd.DataFrame(index=dates)
portfolio_growth['Equal Weight'] = 1 + np.linspace(0, returns_series.dot(weights_equal), len(dates))
portfolio_growth['Market Cap Weight'] = 1 + np.linspace(0, returns_series.dot(weights_mcap), len(dates))
portfolio_growth['Inverse Market Cap'] = 1 + np.linspace(0, returns_series.dot(weights_inverse), len(dates))

portfolio_growth.plot(figsize=(12, 5), title="Simulated Portfolio Growth (Total Return Paths)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value (Indexed to 1)")
plt.grid(True)
plt.tight_layout()
plt.show()
